# Test: lưu một bảng dữ liệu đơn giản lên HDFS

Notebook này:
1. Tạo `SparkSession` nối vào cụm `spark://spark-master:7077`
2. Tạo 1 bảng `nhan_vien` (5 dòng) với schema rõ ràng
3. Ghi xuống HDFS dạng **Parquet** và **CSV**
4. Đọc lại từ HDFS và kiểm tra đúng dữ liệu
5. Liệt kê file đã ghi trên HDFS

Đường dẫn HDFS dùng trong notebook: `/user/jupyter/nhan_vien_*`

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test-luu-bang-len-hdfs")
    .master("spark://spark-master:7077")   # bỏ dòng này => chạy local[*]
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 07:12:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.0 | master: spark://spark-master:7077


In [2]:
from datetime import date
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType,
)

schema = StructType([
    StructField("id",       IntegerType(), False),
    StructField("ho_ten",   StringType(),  False),
    StructField("phong_ban", StringType(), True),
    StructField("luong",    DoubleType(),  True),
    StructField("ngay_vao", DateType(),    True),
])

data = [
    (1, "Nguyen Van A", "Data",       1500.0, date(2023, 1, 10)),
    (2, "Tran Thi B",   "Data",       1800.0, date(2022, 6, 1)),
    (3, "Le Van C",     "Engineering", 2100.0, date(2021, 3, 15)),
    (4, "Pham Thi D",   "Engineering", 1950.0, date(2024, 2, 20)),
    (5, "Hoang Van E",  "Analytics",   1700.0, date(2023, 9, 5)),
]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()
print("So dong:", df.count())

+---+------------+-----------+------+----------+
| id|      ho_ten|  phong_ban| luong|  ngay_vao|
+---+------------+-----------+------+----------+
|  1|Nguyen Van A|       Data|1500.0|2023-01-10|
|  2|  Tran Thi B|       Data|1800.0|2022-06-01|
|  3|    Le Van C|Engineering|2100.0|2021-03-15|
|  4|  Pham Thi D|Engineering|1950.0|2024-02-20|
|  5| Hoang Van E|  Analytics|1700.0|2023-09-05|
+---+------------+-----------+------+----------+

root
 |-- id: integer (nullable = false)
 |-- ho_ten: string (nullable = false)
 |-- phong_ban: string (nullable = true)
 |-- luong: double (nullable = true)
 |-- ngay_vao: date (nullable = true)

So dong: 5


## Ghi xuống HDFS

`mode("overwrite")` = ghi đè nếu đã tồn tại. Kết quả là 1 **thư mục** trên HDFS
chứa các file `part-*` (Spark ghi song song nhiều phần).

In [3]:
# 1) Parquet (định dạng cột, nén tốt, giữ nguyên kiểu dữ liệu) — nên dùng
(df.write
   .mode("overwrite")
   .parquet("hdfs://hadoop-namenode:9000/user/jupyter/nhan_vien_parquet"))

print("Da ghi Parquet -> /user/jupyter/nhan_vien_parquet")

Da ghi Parquet -> /user/jupyter/nhan_vien_parquet


In [4]:
# 2) CSV (dễ đọc bằng mắt / Excel) — gộp về 1 file cho gọn bằng coalesce(1)
(df.coalesce(1).write
   .mode("overwrite")
   .option("header", True)
   .csv("hdfs://hadoop-namenode:9000/user/jupyter/nhan_vien_csv"))

print("Da ghi CSV -> /user/jupyter/nhan_vien_csv")

Da ghi CSV -> /user/jupyter/nhan_vien_csv


## Đọc lại từ HDFS và kiểm tra

In [9]:
back = spark.read.parquet("hdfs://hadoop-namenode:9000/user/jupyter/nhan_vien_parquet")
back.orderBy("id").show()

assert back.count() == df.count(), "So dong khong khop!"
assert set(back.columns) == set(df.columns), "Cot khong khop!"
print("OK - doc lai tu HDFS dung", back.count(), "dong,", len(back.columns), "cot")

+---+------------+-----------+------+----------+
| id|      ho_ten|  phong_ban| luong|  ngay_vao|
+---+------------+-----------+------+----------+
|  1|Nguyen Van A|       Data|1500.0|2023-01-10|
|  2|  Tran Thi B|       Data|1800.0|2022-06-01|
|  3|    Le Van C|Engineering|2100.0|2021-03-15|
|  4|  Pham Thi D|Engineering|1950.0|2024-02-20|
|  5| Hoang Van E|  Analytics|1700.0|2023-09-05|
+---+------------+-----------+------+----------+

OK - doc lai tu HDFS dung 5 dong, 5 cot


In [6]:
# Thử 1 phép tổng hợp trên dữ liệu vừa đọc từ HDFS
from pyspark.sql import functions as F

(back.groupBy("phong_ban")
     .agg(F.count("*").alias("so_nguoi"), F.round(F.avg("luong"), 1).alias("luong_tb"))
     .orderBy("phong_ban")
     .show())

+-----------+--------+--------+
|  phong_ban|so_nguoi|luong_tb|
+-----------+--------+--------+
|  Analytics|       1|  1700.0|
|       Data|       2|  1650.0|
|Engineering|       2|  2025.0|
+-----------+--------+--------+



## Liệt kê file đã ghi trên HDFS (dùng Hadoop FileSystem API qua Spark)

In [7]:
hadoop = spark._jvm.org.apache.hadoop
fs = hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())

for path in ["/user/jupyter/nhan_vien_parquet", "/user/jupyter/nhan_vien_csv"]:
    print(path)
    for st in fs.listStatus(hadoop.fs.Path(path)):
        print(f"   {st.getLen():>8} bytes  {st.getPath().getName()}")

/user/jupyter/nhan_vien_parquet
          0 bytes  _SUCCESS
       1509 bytes  part-00000-9566c9c0-6d65-4658-89fe-847290e041f8-c000.snappy.parquet
       1570 bytes  part-00001-9566c9c0-6d65-4658-89fe-847290e041f8-c000.snappy.parquet
/user/jupyter/nhan_vien_csv
          0 bytes  _SUCCESS
        235 bytes  part-00000-f0dc0d1f-e195-4f0a-a02c-1b9d3ecdbffb-c000.csv


Kiểm tra thêm bằng terminal ngoài Docker:
```bash
docker exec hadoop-namenode hdfs dfs -ls -R /user/jupyter
docker exec hadoop-namenode hdfs dfs -cat /user/jupyter/nhan_vien_csv/part-*.csv
```
Hoặc mở http://localhost:9870 -> Utilities -> Browse the file system -> `/user/jupyter`

In [ ]:
# (Tuy chon) Xoa du lieu test tren HDFS
# fs.delete(hadoop.fs.Path("/user/jupyter"), True)

spark.stop()
print("Done.")